### Build a Simple LLM Application with LCEL

In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this video, you'll have a high level overview of:

Using language models

Using PromptTemplates and OutputParsers

Using LangChain Expression Language (LCEL) to chain components together

Debugging and tracing your application using LangSmith

Deploying your application with LangServe

In [17]:
## Open AI API Key and Open Source models--llama3,Gemma2,mistral--Groq

import os
from dotenv import load_dotenv
load_dotenv("/Users/xe/Documents/LANGCHAIN_PROJECT/.env", override=True)

import openai
openai.api_key=os.getenv("OPENAI_API_KEY")
groq_api_key=os.getenv("GROQ_API_KEY")
groq_api_key

'YOUR_GROQ_API_KEY'

In [18]:
%pip install langchain_groq


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [19]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.3-70b-versatile",groq_api_key=groq_api_key)
model


ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x157c4a450>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x161813440>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [20]:
%pip install langchain_core


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [21]:
from langchain_core.messages import HumanMessage,SystemMessage

messages=[
    SystemMessage(content="Translate the following English to Japanese"),
    HumanMessage(content="Hello, How are you?")
]

result=model.invoke(messages)
result

AIMessage(content='（Konnichiwa, Ogenki desu ka?）', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 47, 'total_tokens': 62, 'completion_time': 0.062854273, 'completion_tokens_details': None, 'prompt_time': 0.011332346, 'prompt_tokens_details': None, 'queue_time': 0.150979475, 'total_time': 0.074186619}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d317b-9ba3-7f33-be65-86f77d46157c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, 'output_tokens': 15, 'total_tokens': 62})

In [22]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
parser.invoke(result)


'（Konnichiwa, Ogenki desu ka?）'

In [23]:
## Using LCEL- chain the components
chain=model| parser
chain.invoke(messages)

'こんにちは、元気ですか？(Konnichiwa, genki desu ka?)'

In [27]:
## prompt Template 
from langchain_core.prompts import ChatPromptTemplate

genric_template="Transalte the following into {language}"

prompt=ChatPromptTemplate.from_messages(
    [("system",genric_template),("user","{text}")]
)


In [35]:
prompt.invoke({"language":"French","text":"Hello, How are you?"})

ChatPromptValue(messages=[SystemMessage(content='Transalte the following into French', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello, How are you?', additional_kwargs={}, response_metadata={})])

In [36]:
result=prompt.invoke({"language":"French","text":"Hello, How are you?"})

In [37]:
result.to_messages()

[SystemMessage(content='Transalte the following into French', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello, How are you?', additional_kwargs={}, response_metadata={})]

In [38]:
chain=prompt|model|parser
chain.invoke({"language":"French","text":"Hello, How are you?"})

'Bonjour, comment allez-vous ?'